In [2]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.ui.port", "4040")
    #.set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.0.0,io.delta:delta-spark_2.12:3.3.2")
    #.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    #.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    #.set("spark.executor.heartbeatInterval", "300000")
    #.set("spark.network.timeout", "400000")
    #.set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    #.set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    #.set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    #.set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

spark = (
    SparkSession.builder.master("local[*]").
        appName('spark-sql-notebook').
        config(conf=sparkConf).
        getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 16:49:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [23]:
# sample data for dataframe
studentMarks = [
    (101, "Alice", "Math", 85),
    (101, "Alice", "Chemistry", 85),
    (101, "Alice", "Physics", 78),
    (101, "Alice", "Sanskrit", 76),
    (101, "Alice", "Hindi", 75),
    (102, "Bob", "Math", 90),
    (102, "Bob", "Chemistry", 85),
    (102, "Bob", "Physics", 88),
    (102, "Bob", "Sanskrit", 74),
    (102, "Bob", "Hindi", 76),
    (103, "Charlie", "Math", 75),
    (103, "Charlie", "Chemistry", 75),
    (103, "Charlie", "Physics", 78),
    (103, "Charlie", "Sanskrit", 72),
    (103, "Charlie", "Hindi", 77),
    (104, "David", "Math", 88),
    (104, "David", "Chemistry", 88),
    (104, "David", "Physics", 95),
    (104, "David", "Sanskrit", 72),
    (104, "David", "Hindi", 77),
    (105, "Eve", "Math", 88),
    (105, "Eve", "Chemistry", 91),
    (105, "Eve", "Physics", 95),
    (105, "Eve", "Sanskrit", 85),
    (105, "Eve", "Hindi", 80)
]

# column names for dataframe
columns = ["Roll_No", "Student_Name", "Subject", "Marks"]

df = spark.createDataFrame(data=studentMarks, schema=columns)
df.show()



+-------+------------+---------+-----+
|Roll_No|Student_Name|  Subject|Marks|
+-------+------------+---------+-----+
|    101|       Alice|     Math|   85|
|    101|       Alice|Chemistry|   85|
|    101|       Alice|  Physics|   78|
|    101|       Alice| Sanskrit|   76|
|    101|       Alice|    Hindi|   75|
|    102|         Bob|     Math|   90|
|    102|         Bob|Chemistry|   85|
|    102|         Bob|  Physics|   88|
|    102|         Bob| Sanskrit|   74|
|    102|         Bob|    Hindi|   76|
|    103|     Charlie|     Math|   75|
|    103|     Charlie|Chemistry|   75|
|    103|     Charlie|  Physics|   78|
|    103|     Charlie| Sanskrit|   72|
|    103|     Charlie|    Hindi|   77|
|    104|       David|     Math|   88|
|    104|       David|Chemistry|   88|
|    104|       David|  Physics|   95|
|    104|       David| Sanskrit|   72|
|    104|       David|    Hindi|   77|
+-------+------------+---------+-----+
only showing top 20 rows



#### Window Aggregate Functions

In [ ]:
from pyspark.sql.functions import *

# Using filter on aggregate data
df.groupBy("Roll_No", "Student_Name") \
    .agg(
        sum("Marks").alias("Total_Marks"), \
        avg("Marks").alias("Average_Marks"), \
        min("Marks").alias("Min_Marks"), \
        max("Marks").alias("Max_Marks")) \
    .show(truncate=False)

+-------+------------+-----------+-------------+---------+---------+
|Roll_No|Student_Name|Total_Marks|Average_Marks|Min_Marks|Max_Marks|
+-------+------------+-----------+-------------+---------+---------+
|101    |Alice       |399        |79.8         |75       |85       |
|102    |Bob         |413        |82.6         |74       |90       |
|103    |Charlie     |377        |75.4         |72       |78       |
|104    |David       |420        |84.0         |72       |95       |
|105    |Eve         |439        |87.8         |80       |95       |
+-------+------------+-----------+-------------+---------+---------+



In [ ]:
# importing Window from pyspark.sql.window
from pyspark.sql.window import Window
#
from pyspark.sql.functions import *

# creating a window partition of dataframe
windowSpec  = Window.partitionBy("Roll_No", "Student_Name").orderBy(desc("Marks"))




""" Aggregate Functions """
df.withColumn("Avg", avg(col("Marks")).over(windowSpec)) \
    .withColumn("Sum", sum(col("Marks")).over(windowSpec)) \
    .withColumn("Min", min(col("Marks")).over(windowSpec)) \
    .withColumn("Max", max(col("Marks")).over(windowSpec)) \
    .show()

+-------+------------+---------+-----+----+---+---+---+
|Roll_No|Student_Name|  Subject|Marks| Avg|Sum|Min|Max|
+-------+------------+---------+-----+----+---+---+---+
|    101|       Alice|     Math|   85|79.8|399| 75| 85|
|    101|       Alice|Chemistry|   85|79.8|399| 75| 85|
|    101|       Alice|  Physics|   78|79.8|399| 75| 85|
|    101|       Alice| Sanskrit|   76|79.8|399| 75| 85|
|    101|       Alice|    Hindi|   75|79.8|399| 75| 85|
|    102|         Bob|     Math|   90|82.6|413| 74| 90|
|    102|         Bob|Chemistry|   85|82.6|413| 74| 90|
|    102|         Bob|  Physics|   88|82.6|413| 74| 90|
|    102|         Bob| Sanskrit|   74|82.6|413| 74| 90|
|    102|         Bob|    Hindi|   76|82.6|413| 74| 90|
|    103|     Charlie|     Math|   75|75.4|377| 72| 78|
|    103|     Charlie|Chemistry|   75|75.4|377| 72| 78|
|    103|     Charlie|  Physics|   78|75.4|377| 72| 78|
|    103|     Charlie| Sanskrit|   72|75.4|377| 72| 78|
|    103|     Charlie|    Hindi|   77|75.4|377| 

#### Window Ranking Functions

##### row_number


In [29]:
# importing row_number() from pyspark.sql.functions
from pyspark.sql.functions import row_number

# creating a window partition of dataframe
windowSpec  = Window.partitionBy("Roll_No", "Student_Name").orderBy(desc("Marks"))

# applying the row_number() function
df.withColumn("row_number", row_number().over(windowSpec)).show()

+-------+------------+---------+-----+----------+
|Roll_No|Student_Name|  Subject|Marks|row_number|
+-------+------------+---------+-----+----------+
|    101|       Alice|     Math|   85|         1|
|    101|       Alice|Chemistry|   85|         2|
|    101|       Alice|  Physics|   78|         3|
|    101|       Alice| Sanskrit|   76|         4|
|    101|       Alice|    Hindi|   75|         5|
|    102|         Bob|     Math|   90|         1|
|    102|         Bob|  Physics|   88|         2|
|    102|         Bob|Chemistry|   85|         3|
|    102|         Bob|    Hindi|   76|         4|
|    102|         Bob| Sanskrit|   74|         5|
|    103|     Charlie|  Physics|   78|         1|
|    103|     Charlie|    Hindi|   77|         2|
|    103|     Charlie|     Math|   75|         3|
|    103|     Charlie|Chemistry|   75|         4|
|    103|     Charlie| Sanskrit|   72|         5|
|    104|       David|  Physics|   95|         1|
|    104|       David|     Math|   88|         2|



##### rank


In [16]:
# importing rank() from pyspark.sql.functions
from pyspark.sql.functions import rank

# applying the rank() function
df.withColumn("rank", rank().over(windowSpec)) \
    .show()

+-------+------------+---------+-----+----+
|Roll_No|Student_Name|  Subject|Marks|rank|
+-------+------------+---------+-----+----+
|    101|       Alice|     Math|   85|   1|
|    101|       Alice|Chemistry|   85|   1|
|    101|       Alice|  Physics|   78|   3|
|    101|       Alice| Sanskrit|   76|   4|
|    101|       Alice|    Hindi|   75|   5|
|    102|         Bob|     Math|   90|   1|
|    102|         Bob|  Physics|   88|   2|
|    102|         Bob|Chemistry|   85|   3|
|    102|         Bob|    Hindi|   76|   4|
|    102|         Bob| Sanskrit|   74|   5|
|    103|     Charlie|  Physics|   78|   1|
|    103|     Charlie|    Hindi|   77|   2|
|    103|     Charlie|     Math|   75|   3|
|    103|     Charlie|Chemistry|   75|   3|
|    103|     Charlie| Sanskrit|   72|   5|
|    104|       David|  Physics|   95|   1|
|    104|       David|     Math|   88|   2|
|    104|       David|Chemistry|   88|   2|
|    104|       David|    Hindi|   77|   4|
|    104|       David| Sanskrit|

##### percent_rank


In [17]:
# importing percent_rank() from pyspark.sql.functions
from pyspark.sql.functions import percent_rank

# applying the percent_rank() function
df.withColumn("percent_rank", percent_rank().over(windowSpec)) \
    .show()

+-------+------------+---------+-----+------------+
|Roll_No|Student_Name|  Subject|Marks|percent_rank|
+-------+------------+---------+-----+------------+
|    101|       Alice|     Math|   85|         0.0|
|    101|       Alice|Chemistry|   85|         0.0|
|    101|       Alice|  Physics|   78|         0.5|
|    101|       Alice| Sanskrit|   76|        0.75|
|    101|       Alice|    Hindi|   75|         1.0|
|    102|         Bob|     Math|   90|         0.0|
|    102|         Bob|  Physics|   88|        0.25|
|    102|         Bob|Chemistry|   85|         0.5|
|    102|         Bob|    Hindi|   76|        0.75|
|    102|         Bob| Sanskrit|   74|         1.0|
|    103|     Charlie|  Physics|   78|         0.0|
|    103|     Charlie|    Hindi|   77|        0.25|
|    103|     Charlie|     Math|   75|         0.5|
|    103|     Charlie|Chemistry|   75|         0.5|
|    103|     Charlie| Sanskrit|   72|         1.0|
|    104|       David|  Physics|   95|         0.0|
|    104|   

##### dense_rank

In [18]:
# importing dense_rank() from pyspark.sql.functions
from pyspark.sql.functions import dense_rank

# applying the dense_rank() function
df.withColumn("dense_rank", dense_rank().over(windowSpec)) \
    .show()

+-------+------------+---------+-----+----------+
|Roll_No|Student_Name|  Subject|Marks|dense_rank|
+-------+------------+---------+-----+----------+
|    101|       Alice|     Math|   85|         1|
|    101|       Alice|Chemistry|   85|         1|
|    101|       Alice|  Physics|   78|         2|
|    101|       Alice| Sanskrit|   76|         3|
|    101|       Alice|    Hindi|   75|         4|
|    102|         Bob|     Math|   90|         1|
|    102|         Bob|  Physics|   88|         2|
|    102|         Bob|Chemistry|   85|         3|
|    102|         Bob|    Hindi|   76|         4|
|    102|         Bob| Sanskrit|   74|         5|
|    103|     Charlie|  Physics|   78|         1|
|    103|     Charlie|    Hindi|   77|         2|
|    103|     Charlie|     Math|   75|         3|
|    103|     Charlie|Chemistry|   75|         3|
|    103|     Charlie| Sanskrit|   72|         4|
|    104|       David|  Physics|   95|         1|
|    104|       David|     Math|   88|         2|


### Window Analytics Functions

##### cume_dist 

In [19]:
# importing cume_dist()
# from pyspark.sql.functions
from pyspark.sql.functions import cume_dist

# applying window function with the help of DataFrame.withColumn
df.withColumn("cume_dist", cume_dist().over(windowSpec)).show()

+-------+------------+---------+-----+---------+
|Roll_No|Student_Name|  Subject|Marks|cume_dist|
+-------+------------+---------+-----+---------+
|    101|       Alice|     Math|   85|      0.4|
|    101|       Alice|Chemistry|   85|      0.4|
|    101|       Alice|  Physics|   78|      0.6|
|    101|       Alice| Sanskrit|   76|      0.8|
|    101|       Alice|    Hindi|   75|      1.0|
|    102|         Bob|     Math|   90|      0.2|
|    102|         Bob|  Physics|   88|      0.4|
|    102|         Bob|Chemistry|   85|      0.6|
|    102|         Bob|    Hindi|   76|      0.8|
|    102|         Bob| Sanskrit|   74|      1.0|
|    103|     Charlie|  Physics|   78|      0.2|
|    103|     Charlie|    Hindi|   77|      0.4|
|    103|     Charlie|     Math|   75|      0.8|
|    103|     Charlie|Chemistry|   75|      0.8|
|    103|     Charlie| Sanskrit|   72|      1.0|
|    104|       David|  Physics|   95|      0.2|
|    104|       David|     Math|   88|      0.6|
|    104|       Davi

##### lag

`GET AND USE PREVIOUS ROW (LAG)`
LAG is a function in SQL which is used to access previous row values in the current row. This is useful when we have use cases like comparison with the previous value. LAG in Spark data frames is available in Window functions

In [ ]:
# importing lag() from pyspark.sql.functions
from pyspark.sql.functions import lag

# to get previous highest
df.withColumn("Lag", lag("Marks", 1).over(windowSpec)) \
    .show()

+-------+------------+---------+-----+----+
|Roll_No|Student_Name|  Subject|Marks| Lag|
+-------+------------+---------+-----+----+
|    101|       Alice|     Math|   85|NULL|
|    101|       Alice|Chemistry|   85|  85|
|    101|       Alice|  Physics|   78|  85|
|    101|       Alice| Sanskrit|   76|  78|
|    101|       Alice|    Hindi|   75|  76|
|    102|         Bob|     Math|   90|NULL|
|    102|         Bob|  Physics|   88|  90|
|    102|         Bob|Chemistry|   85|  88|
|    102|         Bob|    Hindi|   76|  85|
|    102|         Bob| Sanskrit|   74|  76|
|    103|     Charlie|  Physics|   78|NULL|
|    103|     Charlie|    Hindi|   77|  78|
|    103|     Charlie|     Math|   75|  77|
|    103|     Charlie|Chemistry|   75|  75|
|    103|     Charlie| Sanskrit|   72|  75|
|    104|       David|  Physics|   95|NULL|
|    104|       David|     Math|   88|  95|
|    104|       David|Chemistry|   88|  88|
|    104|       David|    Hindi|   77|  88|
|    104|       David| Sanskrit|

##### lead

`GET AND USE NEXT ROW (LEAD)`
LEAD is a function in SQL which is used to access next row values in the current row. This is useful when we have use-cases like comparison with the next value. LEAD in Spark data frames is available in Window functions

In [ ]:
# importing lead() from pyspark.sql.functions
from pyspark.sql.functions import lead

# to get next lowest
df.withColumn("Lead", lead("Marks", 2).over(windowSpec)) \
    .show()

+-------+------------+---------+-----+----+
|Roll_No|Student_Name|  Subject|Marks|Lead|
+-------+------------+---------+-----+----+
|    101|       Alice|     Math|   85|  78|
|    101|       Alice|Chemistry|   85|  76|
|    101|       Alice|  Physics|   78|  75|
|    101|       Alice| Sanskrit|   76|NULL|
|    101|       Alice|    Hindi|   75|NULL|
|    102|         Bob|     Math|   90|  85|
|    102|         Bob|  Physics|   88|  76|
|    102|         Bob|Chemistry|   85|  74|
|    102|         Bob|    Hindi|   76|NULL|
|    102|         Bob| Sanskrit|   74|NULL|
|    103|     Charlie|  Physics|   78|  75|
|    103|     Charlie|    Hindi|   77|  75|
|    103|     Charlie|     Math|   75|  72|
|    103|     Charlie|Chemistry|   75|NULL|
|    103|     Charlie| Sanskrit|   72|NULL|
|    104|       David|  Physics|   95|  88|
|    104|       David|     Math|   88|  77|
|    104|       David|Chemistry|   88|  72|
|    104|       David|    Hindi|   77|NULL|
|    104|       David| Sanskrit|